In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from util import project_points, scale_intrinsics, inv2x2

In [ ]:
def gaussian_rasterization(pos, color, opacity, sigma, c2w, H, W, fx, fy,
                           cx, cy, near=2e-3, far=100, pix_guard=64, 
                           T=16, min_conis=1e-6, chi_square_clip=9.21,
                           alpha_max=0.999, alpha_cutoff=1/255.):
    uv, x, y, z = project_points(pos, c2w, H, W, fx, fy, cx, cy)
    
    device = pos.device
    dt = pos.dtype
    
    u, v = uv[:, 0], uv[:, 1]
    frustum = (
        (u > -pix_guard) 
        & (u < W + pix_guard) 
        & (v > -pix_guard) 
        & (v < H + pix_guard) 
        & (z > near) 
        & (z < far)
    )
    uv = uv[frustum]
    pos = pos[frustum]
    color = color[frustum]
    opacity = torch.sigmoid(opacity[frustum]).clamp(0, 0.999)
    z = z[frustum]
    sigma = sigma[frustum]
    
    ## Project the covariance
    Rcw = c2w[:3, :3]
    Rwc = Rcw.T
    
    # Eq. 5
    J = torch.zeros((pos.shape[0], 2, 3), device=device, dtype=dt)
    J[:, 0, 0] = fx / z
    J[:, 1, 1] = fy / z
    J[:, 0, 2] = -fx * x / (z * z)
    J[:, 1, 2] = -fy * y / (z * z)
    
    sigma_camera = Rwc.unsqueeze(0) @ sigma @ Rwc.T.unsqueeze(0)
    sigma_uv = J @ sigma_camera @ J.transpose(1, 2)
    
    # Enforce symmetry
    sigma_uv = 0.5 * (sigma_uv + sigma_uv.transpose(1, 2))
    
    # Clamp projected gaussian ellipse size
    evals, evecs = torch.linalg.eigh(sigma_uv)
    evals = torch.clamp(evals, min=1e-6, max=1e4)
    sigma_uv = evecs @ torch.diag(evals) @ evecs.transpose(1, 2)
    
    # Filter NaN and infinity
    keep = torch.isfinite(sigma_uv.reshape(sigma_uv.shape[0], -1)).all(dim=-1)
    
    uv = uv[keep]
    pos = pos[keep]
    color = color[keep]
    opacity = opacity[keep]
    z = z[keep]
    sigma_uv = sigma_uv[keep]
    
    order = torch.sort(z, descending=False)
    uv = uv[order]
    color = color[order]
    opacity = opacity[order]
    sigma_uv = sigma_uv[order]
    
    u = uv[:, 0]
    v = uv[:, 1]
    
    # Tiling
    TODO = None
    
    major_variance = evals[:, 1].clamp(min=1e-12, max=1e4) # [N]
    radius = 3.0 * torch.sqrt(major_variance)
    
    umin = torch.floor(u - radius)
    umax = torch.ceil(u + radius)
    vmin = torch.floor(v - radius)
    vmax = torch.ceil(v + radius)
    
    on_screen = (umax >= 0) & (umin < W) & (vmax >= 0) & (vmin < H)
    if not on_screen.any():
        raise Exception("there are no gaussians on screen")
    
    u, v = u[on_screen], v[on_screen]
    color = color[on_screen]
    opacity = opacity[on_screen]
    sigma_camera = sigma_camera[on_screen]
    umin, umax = umin[on_screen], umax[on_screen]
    vmin, vmax = vmin[on_screen], vmax[on_screen]
    
    umin = umin.clamp(0, W - 1)
    umax = umax.clamp(0, W - 1)
    vmin = vmin.clamp(0, H - 1)
    vmax = vmax.clamp(0, H - 1)
    
    # Tile index for each AABB
    umin_tile = (umin // T).to(torch.int64) # [N]
    umax_tile = (umax // T).to(torch.int64)
    vmin_tile = (vmin // T).to(torch.int64)
    vmax_tile = (vmax // T).to(torch.int64)
    
    # Number of tiles each haussian intersects
    n_u = umax_tile - umin_tile + 1  # [N] [3, 4, 8, 1, 7, 5]
    n_v = vmax_tile - vmin_tile + 1  # [N] [1, 2, 6, 1, 3, 4]
    
    """
    n_u reshaped
    [[3], [4], [8], [1], [7], [5]]
    n_u.unsqueeze(-1) == n_u[:, None]
    
    span_indices_u reshaped
    [[1, 2, 3, 4, 5, 6, 7, 8]]
    span_indices_u.unsqueeze(0) == span_indices_u[None]
    """
    
    # Max number of tiles
    max_u = int(n_u.max().item())
    max_v = int(n_v.max().item())
    
    span_indices_u = torch.arange(max_u, device=device, dtype=torch.int64) # [max_u] [1, 2, 3, 4, 5, 6, 7, 8]
    span_indices_v = torch.arange(max_v, device=device, dtype=torch.int64) # [max_v] [1, 2, 3, 4, 5, 6]
    tile_u = (umin_tile.unsqueeze(-1) + span_indices_u.unsqueeze(0))[:, :, None] # [N, max_u, 1]
    tile_v = (vmin_tile.unsqueeze(-1) + span_indices_v.unsqueeze(0))[:, None, :] # [N, 1, max_v]
    
    compact_mask = (span_indices_u[None, :, None] < n_u[:, None, None]
                    ) & (span_indices_v[None, None, :] < n_v[:, None, None]) # [N, max_u, max_v]
    flat_tile_u = tile_u[compact_mask]
    flat_tile_v = tile_v[compact_mask]
    
    gaussian_ids = 0
    
    inverse_covariance = inv2x2(sigma_camera)
    inverse_covariance[:, 0, 0] = torch.clamp(inverse_covariance[:, 0, 0], min=min_conis)
    inverse_covariance[:, 1, 1] = torch.clamp(inverse_covariance[:, 1, 1], min=min_conis)
    
    final_image = torch.zeros((H * W, 3), device=device, dtype=dt)
    
    # Iterate over tiles
    for ids in TODO:
        txi = TODO
        tyi = TODO
        
        x0, y0 = txi * T, tyi * T
        x1, y1 = min((txi + 1) * T, W), min((tyi + 1) * T, H)
        if x0 >= x1 or y0 >= y1:
            continue
        
        xs = torch.arange(x0, x1, device=device, dtype=dt)
        ys = torch.arange(y0, y1, device=device, dtype=dt)
        pu, pv = torch.meshgrid(xs, ys, indexing='xy')
        px_u = pu.reshape(-1)
        px_v = pv.reshape(-1)
        
        pixel_idx_1d = (px_u * W + px_v).to(torch.int64)
        
        gaussian_i_u = u[ids]
        gaussian_i_v = v[ids]
        gaussian_i_color = color[ids]
        gaussian_i_opacity = color[ids] # [N]
        gaussian_i_inverse_covariance=inverse_covariance[ids]
        
        du = px_u.unsqueeze(0) - gaussian_i_u.unsqueeze(-1) # [N, T * T]
        dv = px_u.unsqueeze(0) - gaussian_i_v.unsqueeze(-1) # [N, T * T]
        
        A11 = gaussian_i_inverse_covariance[:, 0, 0].unsqueeze(-1) # [N, 1]
        A12 = gaussian_i_inverse_covariance[:, 0, 1].unsqueeze(-1)
        A22 = gaussian_i_inverse_covariance[:, 1, 1].unsqueeze(-1)
        q = A11 * du * du + 2 * A12 * du * dv + A22 * dv * dv # [N, T * T]
        
        inside = q <= chi_square_clip
        g = torch.exp(-0.5 * torch.clamp(q, max=chi_square_clip)) # [N, T * T]
        g = torch.where(inside, g, torch.zeros_like(g))
                
        alpha_i = (gaussian_i_opacity.unsqueeze(-1) * g).clamp(max=alpha_max) # [N, T * T]
        alpha_i = torch.where(alpha_i >= alpha_cutoff, alpha_i, torch.zeros_like(alpha_i))
        one_minus_alpha_i = 1 - alpha_i
        T_i = torch.cumprod(one_minus_alpha_i, dim=0)
        T_i = torch.concatenate([
            torch.ones((1, alpha_i.shape[-1]), device=device, dtype=dt),
            T_i[:-1]
        ], dim=0)
                
        w = alpha_i * T_i
        color = (w.unsqueeze(-1) * gaussian_i_color.unsqueeze(1)).sum(dim=0) # [T * T, 3]
        
        final_image[pixel_idx_1d] = color
        
    return final_image.reshape((H, W, 3)).clamp(0, 1)